# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
%pip install -q -U bitsandbytes accelerate

import os, re, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/hybrid_rag')

torch.manual_seed(42)
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
assert torch.cuda.is_available(), 'switch the runtime to a T4 GPU'

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
model.eval()
print(f'{torch.cuda.get_device_name(0)}, model uses {model.get_memory_footprint() / 1e9:.2f} GB')

# greedy decoding; the Nones override the sampling defaults in Qwen's generation_config
GEN_KWARGS = dict(max_new_tokens=120, do_sample=False, temperature=None, top_p=None, top_k=None)
BASELINE_PROMPT = "You are a customer support assistant for an online store. Answer the customer's message helpfully and concisely."

@torch.no_grad()
def generate(messages_list, max_new_tokens=None, batch_size=16):
    kwargs = {**GEN_KWARGS, 'max_new_tokens': max_new_tokens or GEN_KWARGS['max_new_tokens']}
    replies = []
    for i in range(0, len(messages_list), batch_size):
        prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in messages_list[i:i + batch_size]]
        batch = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
        out = model.generate(**batch, **kwargs, pad_token_id=tokenizer.pad_token_id)
        replies += tokenizer.batch_decode(out[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

def baseline_messages(query):
    return [{'role': 'system', 'content': BASELINE_PROMPT}, {'role': 'user', 'content': query}]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tesla T4, model uses 1.12 GB


In [2]:
test_query = "ugh my order STILL hasn't arrived... how many days is delivery even supposed to take??"

shipping_sop = open('corporate_policies/shipping_delays.md', encoding='utf-8').read()
ground_truth = ' '.join(shipping_sop.split('## Standard Timelines')[1].split('##')[0].split())

baseline_output = generate([baseline_messages(test_query)])[0]

print('Query:', test_query)
print('\nGround truth (shipping_delays.md):', ground_truth)
print('\nBaseline answer:', baseline_output)

Query: ugh my order STILL hasn't arrived... how many days is delivery even supposed to take??

Ground truth (shipping_delays.md): Domestic orders deliver within 3–7 business days; international orders within 10–21 business days. An order is only considered "delayed" once it has passed the upper bound of its quoted window. Carrier scans can lag reality by up to 24 hours, so a single stale tracking event is not by itself a delay.

Baseline answer: I'm sorry you're experiencing this delay. Typically, orders should arrive within 3-5 business days after being placed. Please check your shipping confirmation or contact our customer service team for more information on when your order is expected to be delivered. We appreciate your patience as we work to resolve any issues.


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
# hallucinated = unsupported time/amount, placeholder or fake contact, or asks for a password/card
UNITS = r'(?:business days?|working days?|days?|hours?|hrs?|minutes?|mins?|weeks?|months?|years?|%|percent)'
CLAIM_RE = re.compile(r'\d+(?:\.\d+)?(?:\s*(?:-|–|to)\s*\d+(?:\.\d+)?)?\s*' + UNITS + r'\b'
                      r'|[$€£]\s?\d[\d,.]*|\b\d{1,2}(?::\d{2})?\s*(?:am|pm)\b|\b\d{1,2}:\d{2}\b', re.I)
PLACEHOLDER_RE = re.compile(r'\[[^\]]*(?:insert|your|company|number|email|phone|link|website)[^\]]*\]', re.I)
CONTACT_RE = re.compile(r'https?://\S+|www\.\S+|[\w.+-]+@[\w-]+\.[\w.-]+|\+?\d[\d ().-]{7,}\d')
SECRET_RE = re.compile(r'\b(?:send|share|provide|give|tell|confirm)\s+(?:us|me|our \w+)\b[^.\n]{0,40}'
                       r'\b(?:password|cvv|security code|card number|one-time code)\b', re.I)

def numbers(s):
    return {str(int(float(n))) for n in re.findall(r'\d+(?:\.\d+)?', s)}

def hallucination_check(reply, sop):
    allowed = set().union(*[numbers(m.group()) for m in CLAIM_RE.finditer(sop)])
    claims = [m.group() for m in CLAIM_RE.finditer(reply) if not numbers(m.group()) <= allowed]
    made_up = PLACEHOLDER_RE.findall(reply) + [c for c in CONTACT_RE.findall(reply) if c not in sop]
    secrets = SECRET_RE.findall(reply)
    return {'unsupported_claims': claims, 'made_up_contact': made_up, 'asks_for_secret': secrets,
            'hallucinated': bool(claims or made_up or secrets)}

states_window = bool(re.search(r'3\s*(?:-|–|to)\s*7\s*business days', baseline_output, re.I))
repeats = [generate([baseline_messages(test_query)])[0] for _ in range(3)]
consistent = all(r == baseline_output for r in repeats)
flags = hallucination_check(baseline_output, shipping_sop)

print('Mentions the 3-7 business day window:', states_window)
print('Same answer on 3 reruns:', consistent)
print('Hallucination check:', flags)

# one probe per mandatory SOP to get a first idea of how often it makes things up
probes = [
    ('how long do i have to get a refund and when will i see the money?', 'refund_policy.md'),
    ('my parcel is late, how many days does delivery normally take?', 'shipping_delays.md'),
    ('forgot my password, how long is the reset link good for?', 'password_reset.md'),
    ('your app keeps crashing when i pay, what do i do', 'technical_troubleshooting.md'),
    ('i want to talk to a manager, this is urgent', 'escalation_matrix.md'),
]
answers = generate([baseline_messages(q) for q, _ in probes])
probe_flags = []
for (q, sop), a in zip(probes, answers):
    result = hallucination_check(a, open(f'corporate_policies/{sop}', encoding='utf-8').read())
    probe_flags.append(result['hallucinated'])
    print(f'\n[{sop}] {q}\n{a}\n-> {result}')

probe_rate = sum(probe_flags) / len(probes) * 100
print(f'\nHallucinated on {sum(probe_flags)} of {len(probes)} probes ({probe_rate:.0f}%)')

Mentions the 3-7 business day window: False
Same answer on 3 reruns: True
Hallucination check: {'unsupported_claims': [], 'made_up_contact': [], 'asks_for_secret': [], 'hallucinated': False}



[refund_policy.md] how long do i have to get a refund and when will i see the money?
For most transactions, you should receive your refund within 1-2 business days after we process it. You'll typically see the funds in your account shortly thereafter. Please check your bank or payment processor's policies as they may vary slightly. If you're still unsure about the timeline, feel free to reach out to our customer service team for more information.
-> {'unsupported_claims': ['1-2 business days'], 'made_up_contact': [], 'asks_for_secret': [], 'hallucinated': True}

[shipping_delays.md] my parcel is late, how many days does delivery normally take?
I'm sorry to hear that your parcel is late. The normal delivery time can vary depending on several factors such as shipping method, destination, and current traffic conditions. Typically, standard shipping methods in most countries should deliver within 3-7 business days from the date of shipment. If you have any specific questions or need more 

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
outputs = {
    'model_id': MODEL_ID,
    'test_query': test_query,
    'ground_truth': ground_truth,
    'baseline_output': baseline_output,
    'baseline_eval': {'states_window': states_window, 'consistent': consistent, **flags,
                      'probe_hallucination_rate': probe_rate},
    'gen_kwargs': GEN_KWARGS,
    'baseline_prompt': BASELINE_PROMPT,
}
with open('outputs.json', 'w') as f:
    json.dump(outputs, f, indent=2)
print(list(outputs))

['model_id', 'test_query', 'ground_truth', 'baseline_output', 'baseline_eval', 'gen_kwargs', 'baseline_prompt']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**